# Đánh giá checkpoint variable-rate đã train trên SFU-HW Class D

Notebook này chỉ chạy checkpoint Proposed mới. HEVC/x265, DCVC-RT gốc và SVC được đọc từ các file JSON có sẵn, không mã hóa lại.

Kaggle Inputs cần có:

- SFU-HW Class D: frames/, labels/, manifest.json.
- cvpr2025_image.pth.tar.
- Output train chứa best.pth.tar của random QP, lambda 1..64, 7 frame.
- Ba JSON kết quả: HEVC/x265, Original DCVC-RT, Original SVC.

BD-rate dùng HEVC/x265 làm anchor. SVC chỉ mang tính tham khảo nếu JSON dùng Estimated BPP.


In [ ]:
from pathlib import Path
import json, os, re, shlex, shutil, subprocess, sys
from collections import deque

import torch

assert torch.cuda.is_available(), 'Hãy bật GPU trong Kaggle Settings > Accelerator'
print('GPU:', torch.cuda.get_device_name(0))

KAGGLE_INPUT = Path('/kaggle/input')
KAGGLE_WORKING = Path('/kaggle/working')
QPS_PROPOSED = [0, 10, 21, 32, 42, 53, 63]
RAW_MAP5095_PERCENT = None
RESULTS = KAGGLE_WORKING / 'evaluation_variable_rate'
RESULTS.mkdir(parents=True, exist_ok=True)

assert KAGGLE_INPUT.is_dir()
print('QP Proposed:', QPS_PROPOSED)


In [ ]:
# Clone evaluator đã kiểm chứng; không chạy lại baseline.
PROJECT = KAGGLE_WORKING / 'svc_eval'
YOLO_REPO = KAGGLE_WORKING / 'yolov5'
REPO = 'https://github.com/uetot1/bla.git'
COMMIT = 'b6a5d7de506cddfa3340695b5659c9439f831de5'

if not (PROJECT / '.git').is_dir():
    subprocess.run(['git', 'clone', '--no-checkout', REPO, str(PROJECT)], check=True)
subprocess.run(['git', 'fetch', 'origin', COMMIT, '--depth', '1'], cwd=PROJECT, check=True)
subprocess.run(['git', 'checkout', '--detach', COMMIT], cwd=PROJECT, check=True)

if not (YOLO_REPO / '.git').is_dir():
    subprocess.run([
        'git', 'clone', '--branch', 'v7.0', '--depth', '1',
        'https://github.com/ultralytics/yolov5.git', str(YOLO_REPO),
    ], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'setuptools<81'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PROJECT / 'requirements.txt')], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(YOLO_REPO / 'requirements.txt')], check=True)

setup_py = PROJECT / 'dcvc_rt/src/cpp/setup.py'
setup_text = setup_py.read_text(encoding='utf-8').replace(
    'python_requires=">=3.12"', 'python_requires=">=3.10"'
)
setup_py.write_text(setup_text, encoding='utf-8')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', str(setup_py.parent)], check=True)

os.chdir(PROJECT)
print('Evaluator:', PROJECT)
print('YOLOv5:', YOLO_REPO)


In [ ]:
# Tự tìm dataset, checkpoint train và ba JSON baseline.
manifest_candidates = [
    p for p in KAGGLE_INPUT.rglob('manifest.json')
    if (p.parent / 'frames').is_dir() and (p.parent / 'labels').is_dir()
]
assert len(manifest_candidates) == 1, (
    f'Cần đúng 1 manifest Class D; tìm thấy: {manifest_candidates}'
)
MANIFEST = manifest_candidates[0]
DATA_DIR = MANIFEST.parent


def choose_file(filename):
    matches = list(KAGGLE_INPUT.rglob(filename))
    assert matches, f'Không tìm thấy {filename} trong Kaggle Inputs'
    if len(matches) > 1:
        print(f'Có {len(matches)} file {filename}; dùng:', matches[0])
    return matches[0]


IMAGE_CKPT = choose_file('cvpr2025_image.pth.tar')
YOLO_WEIGHTS = PROJECT / 'yolov5s.pt'

best_candidates = list(KAGGLE_INPUT.rglob('best.pth.tar'))
local_best = (
    KAGGLE_WORKING
    / 'dcvc_rt_vcm_random_qp_lambda_1_64_7frame'
    / 'random_qp_lambda_1_64'
    / 'best.pth.tar'
)
if local_best.is_file():
    best_candidates.insert(0, local_best)

valid_best = []
for path in best_candidates:
    try:
        obj = torch.load(path, map_location='cpu', weights_only=False)
    except Exception:
        continue
    if (
        isinstance(obj, dict)
        and {'p_net', 'student_front'} <= obj.keys()
        and tuple(float(x) for x in obj.get('lambda_range', ())) == (1.0, 64.0)
        and int(obj.get('args', {}).get('p_frames', 6)) == 6
    ):
        valid_best.append((int(obj.get('epoch', -1)), path, obj))

assert valid_best, (
    'Không tìm thấy best.pth.tar của model random-QP lambda 1..64. '
    'Nếu chạy notebook mới, hãy Add Input output của notebook train.'
)
_, TRAINED_CKPT, trained_checkpoint = max(valid_best, key=lambda item: item[0])


def read_result(path):
    try:
        data = json.loads(path.read_text(encoding='utf-8'))
    except Exception:
        return None
    return data if len(data.get('points', [])) >= 4 and data.get('method') else None


result_candidates = [
    (path, data)
    for path in KAGGLE_INPUT.rglob('*.json')
    if (data := read_result(path)) is not None
]


def choose_result(label, predicate):
    matches = [(p, d) for p, d in result_candidates if predicate(d['method'].lower())]
    assert matches, f'Không tìm thấy JSON {label}. Các method: {[d["method"] for _, d in result_candidates]}'
    matches.sort(key=lambda item: len(item[1]['points']), reverse=True)
    if len(matches) > 1:
        print(f'Có {len(matches)} JSON {label}; dùng:', matches[0][0])
    return matches[0][0]


HEVC_JSON = choose_result('HEVC/x265', lambda m: 'hevc' in m or 'x265' in m)
ORIGINAL_JSON = choose_result(
    'Original DCVC-RT',
    lambda m: 'original' in m and 'dcvc' in m and 'svc' not in m,
)
SVC_JSON = choose_result(
    'Original SVC',
    lambda m: 'svc' in m and 'dcvc' not in m,
)

print('Dataset:', DATA_DIR)
print('Image checkpoint:', IMAGE_CKPT)
print('Trained checkpoint:', TRAINED_CKPT)
print('HEVC JSON:', HEVC_JSON)
print('DCVC-RT JSON:', ORIGINAL_JSON)
print('SVC JSON:', SVC_JSON)


In [ ]:
# Chuyển checkpoint train sang checkpoint evaluator gọn.
CONVERTED = KAGGLE_WORKING / 'proposed_variable_rate_eval.pth.tar'
torch.save({
    'schema_version': 7,
    'state_dict': trained_checkpoint['p_net'],
    'cloned_frontend_state_dict': trained_checkpoint['student_front'],
    'hierarchical_qp': True,
    'lambda_range': trained_checkpoint.get('lambda_range', [1.0, 64.0]),
    'lambda_mapping': trained_checkpoint.get('lambda_mapping'),
    'qp_sampling': trained_checkpoint.get('qp_sampling', [0, 63]),
    'trainable_components': ['dcvc_rt_dmc', 'yolo_cloned_frontend'],
    'feature_objective': {
        'task_model': 'yolov5s',
        'cloned_frontend_last_layer': 4,
        'layer_indices': [4],
    },
}, CONVERTED)

assert CONVERTED.is_file()
print('Epoch tốt nhất:', int(trained_checkpoint.get('epoch', -1)) + 1)
print('Checkpoint evaluator:', CONVERTED)


In [ ]:
# Preflight và chạy actual-bitstream evaluation. Rerun sẽ bỏ qua JSON hoàn chỉnh.
import MLCodec_extensions_cpp
from dcvc_rt.src.utils.vcm_eval_dataset import AnnotatedVideoDataset

sequences = list(AnnotatedVideoDataset(DATA_DIR, MANIFEST))
frame_counts = {sequence.name: sequence.frame_count for sequence in sequences}
assert sequences and min(frame_counts.values()) >= 65, frame_counts
assert (YOLO_REPO / 'hubconf.py').is_file()
print('Sequences:', frame_counts)


def safe_name(value):
    return re.sub(r'[^A-Za-z0-9_.-]+', '_', value).strip('_') or 'method'


def result_complete(path, qps):
    if not path.is_file():
        return False
    data = json.loads(path.read_text(encoding='utf-8'))
    saved = data.get('codec_config', {}).get(
        'base_qps', data.get('codec_config', {}).get('qps')
    )
    return list(saved or []) == list(qps) and len(data.get('points', [])) == len(qps)


def run_and_stream(command):
    env = os.environ.copy()
    env['PYTHONPATH'] = str(YOLO_REPO) + os.pathsep + env.get('PYTHONPATH', '')
    process = subprocess.Popen(
        command, cwd=PROJECT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1, env=env,
    )
    tail = deque(maxlen=80)
    for line in process.stdout:
        print(line, end='')
        tail.append(line.rstrip())
    code = process.wait()
    if code:
        raise RuntimeError(f'Command failed ({code}):\n' + '\n'.join(tail))


def run_codec(method, checkpoint, qps):
    output_dir = RESULTS / 'proposed'
    result_path = output_dir / f'{safe_name(method)}_results.json'
    if result_complete(result_path, qps):
        print('SKIP, kết quả đã hoàn chỉnh:', result_path)
        return result_path
    command = [
        sys.executable, 'evaluate_vcm.py', '--mode', 'codec',
        '--data-dir', str(DATA_DIR), '--dataset-manifest', str(MANIFEST),
        '--image-ckpt', str(IMAGE_CKPT), '--video-ckpt', str(checkpoint),
        '--qps', *map(str, qps), '--reset-interval', '32',
        '--minimum-sequence-frames', '65', '--force-zero-thres', '0.12',
        '--codec-precision', 'fp16', '--yolov5-repo', str(YOLO_REPO),
        '--yolov5-weights', str(YOLO_WEIGHTS), '--detector-size', '640',
        '--confidence-threshold', '0.001', '--nms-iou-threshold', '0.6',
        '--max-detections', '300', '--method-name', method,
        '--output-dir', str(output_dir),
        '--bitstream-dir', str(KAGGLE_WORKING / 'vcm_bitstreams'),
    ]
    print('RUN:', shlex.join(command))
    run_and_stream(command)
    assert result_complete(result_path, qps), result_path
    return result_path


## Chạy Proposed

Đây là phần duy nhất phải mã hóa/giải mã lại. Mỗi QP xử lý toàn bộ Class D nên progress đầu tiên có thể đứng ở 0/4 khá lâu.


In [ ]:
PROPOSED_JSON = run_codec(
    'Proposed DCVC-RT-VCM Variable-Rate',
    CONVERTED,
    QPS_PROPOSED,
)
print('Proposed JSON:', PROPOSED_JSON)


## So sánh RD và BD-rate

HEVC/x265 là anchor duy nhất. BD-rate âm nghĩa là candidate tiết kiệm bitrate tại cùng mAP.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.interpolate import PchipInterpolator
from IPython.display import display
from dcvc_rt.src.utils.bd_rate import compute_bd_rate, pareto_front


def load_result(path):
    data = json.loads(Path(path).read_text(encoding='utf-8'))
    assert data.get('schema_version') in (6, 7)
    assert len(data.get('points', [])) >= 4
    return data


proposed = load_result(PROPOSED_JSON)
original = load_result(ORIGINAL_JSON)
hevc = load_result(HEVC_JSON)
svc = load_result(SVC_JSON)

compatibility_keys = (
    'evaluation_id', 'task_model', 'protocol', 'ground_truth',
    'detector_config', 'comparison_scope',
)
for data in (proposed, original, hevc):
    for key in compatibility_keys:
        assert json.dumps(data.get(key), sort_keys=True) == json.dumps(
            original.get(key), sort_keys=True
        ), f'Protocol lệch ở {key}: {data["method"]}'
    assert {int(p['coded_frames']) for p in data['points']} == {
        int(p['coded_frames']) for p in original['points']
    }, f'Số frame lệch: {data["method"]}'

assert svc['evaluation_id'] == original['evaluation_id'], 'SVC dùng khác dataset/manifest'
assert {int(p['coded_frames']) for p in svc['points']} == {
    int(p['coded_frames']) for p in original['points']
}, 'SVC dùng khác số frame'

curves = [proposed, original, hevc, svc]
metric = 'map5095'
rate_keys = {
    data['method']: (
        'estimated_bpp'
        if data is svc and 'estimated_bpp' in data['points'][0]
        else 'actual_bpp'
    )
    for data in curves
}
rate_sources = {
    data['method']: ('estimated' if rate_keys[data['method']] == 'estimated_bpp' else 'actual')
    for data in curves
}
pareto = {}
for data in curves:
    rates = [float(p[rate_keys[data['method']]]) for p in data['points']]
    quality = [float(p[metric]) for p in data['points']]
    pareto[data['method']] = pareto_front(rates, quality)

comparison_dir = RESULTS / 'comparison'
comparison_dir.mkdir(parents=True, exist_ok=True)
figure, axis = plt.subplots(figsize=(11, 7))
markers = ('o', 's', '^', 'D')
styles = ('-', '-', '-', '--')
for index, data in enumerate(curves):
    rates, quality = pareto[data['method']]
    quality_grid = np.linspace(quality[0], quality[-1], 250)
    rate_grid = 10 ** PchipInterpolator(quality, np.log10(rates))(quality_grid)
    label = data['method']
    if rate_sources[data['method']] == 'estimated':
        label += ' (Estimated BPP)'
    line = axis.plot(
        rate_grid, quality_grid * 100, linewidth=2.3,
        linestyle=styles[index], label=label,
    )[0]
    axis.scatter(
        rates, quality * 100, marker=markers[index], s=60,
        color=line.get_color(), zorder=3,
    )

if RAW_MAP5095_PERCENT is not None:
    axis.axhline(
        float(RAW_MAP5095_PERCENT), linestyle='--', color='tab:blue',
        label=f'Raw YOLOv5s ({RAW_MAP5095_PERCENT:.2f}%)',
    )
axis.set(
    xlabel='BPP (Actual, except Original SVC if marked Estimated)',
    ylabel='mAP@[0.5:0.95] (%)',
    title='SFU-HW Class D — Object Detection mAP vs BPP',
)
axis.grid(True, alpha=0.3)
axis.legend()
figure.tight_layout()
plot_path = comparison_dir / 'rd_curve_map5095.png'
figure.savefig(plot_path, dpi=220, bbox_inches='tight')
plt.show()

anchor_rate, anchor_quality = pareto[hevc['method']]
bd_rows = []
for candidate in (proposed, original, svc):
    candidate_rate, candidate_quality = pareto[candidate['method']]
    lower = max(float(anchor_quality.min()), float(candidate_quality.min()))
    upper = min(float(anchor_quality.max()), float(candidate_quality.max()))
    candidate_source = rate_sources[candidate['method']]
    bd_rows.append({
        'anchor': hevc['method'],
        'candidate': candidate['method'],
        'metric': metric,
        'anchor_rate_source': 'actual',
        'candidate_rate_source': candidate_source,
        'validity': (
            'comparable_actual_bpp'
            if candidate_source == 'actual'
            else 'exploratory_mixed_actual_estimated'
        ),
        'overlap_mAP_min_percent': lower * 100,
        'overlap_mAP_max_percent': upper * 100,
        'bd_rate_percent': (
            compute_bd_rate(
                anchor_rate, anchor_quality,
                candidate_rate, candidate_quality,
            )
            if lower < upper else None
        ),
        'interpretation': 'negative means bitrate saving vs HEVC/x265',
    })

bd_frame = pd.DataFrame(bd_rows)
display(bd_frame)
bd_frame.to_csv(comparison_dir / 'bd_rate_vs_hevc.csv', index=False)
(comparison_dir / 'bd_rate_vs_hevc.json').write_text(
    json.dumps(bd_rows, indent=2), encoding='utf-8'
)

rd_rows = []
for data in curves:
    key = rate_keys[data['method']]
    for point in data['points']:
        rd_rows.append({
            'method': data['method'],
            'base_qp': point.get('base_qp'),
            'rate_bpp': point[key],
            'rate_source': rate_sources[data['method']],
            'map50': point['map50'],
            'map5095': point['map5095'],
        })
rd_frame = pd.DataFrame(rd_rows)
display(rd_frame)
rd_frame.to_csv(comparison_dir / 'rd_points.csv', index=False)
print('Plot:', plot_path)
print('Kết quả:', comparison_dir)


In [ ]:
# Đóng gói toàn bộ JSON, CSV và hình để tải xuống.
archive = shutil.make_archive(
    str(KAGGLE_WORKING / 'variable_rate_class_d_evaluation'),
    'zip',
    root_dir=RESULTS,
)
print('ZIP:', archive)
